### Meta Data Retrieval 
> Meta data can be used effectively in retrieval of data from knowledge Base  
> It can provide another aspect of search possibility to pull out relevant information  
> Further the semantic search and meta data retrieval can be used in Hybrid mode to improve the context

#### Imports

In [1]:
import lancedb

# Sentence transformers to use the embedding models locally
from sentence_transformers import SentenceTransformer, util
import pandas as pd

# Import required class from Google
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Initialise an client object with API key
load_dotenv ()
client = genai.Client()

**Embedding Models**  
2 differenet models are used for embedding  
The embedding that is needed for vector search shall be teh same model that is used for vector DB creation

In [2]:
Embedder_1 = SentenceTransformer ("sentence-transformers/all-MiniLM-L6-v2",local_files_only=True)
Embedder_2 = SentenceTransformer ("sentence-transformers/all-mpnet-base-v2",local_files_only=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Utility**  
> Function to match and get the relevant meta data from the entire meta data present in vector Db

In [4]:
def Match_Meta_Data (query, meta_data, meta_data_emb, top_k=2):

    # Query    
    query_emb = Embedder_1.encode(query, normalize_embeddings=True)

    # Top-k semantic search
    hits = util.semantic_search (query_embeddings=query_emb, corpus_embeddings=meta_data_emb, top_k=top_k)

    Matches = []
    for hit in hits[0]:

        Matches = Matches + [meta_data[hit['corpus_id']]]
    
    return Matches

> 2 Different Embedding model from Sentence Transformer  
> One used for Search from Vector DB (Vectors in DB and query vector shall be with same embedding model)  
> the other one used for handling meta data

> Connect to the Database that was created

In [5]:
# Connect to existing Vector DB and use data
# Create a Lance DB Vector Base
DB = lancedb.connect ('Vector_DB')

# Create a Table and add the Chunks data
table = DB.open_table ("tech_ref")
print (table.schema)

source: string
topic: string
text: string
vector: fixed_size_list<item: float>[768]
  child 0, item: float


In [6]:
DF= table.to_pandas()
DF

,source,topic,text,vector
0,IBM,Generic,"By , \nStephanie Susnjara \nIan Smalley \n...","[0.04031377, 0.056903712, -0.034424566, 0.0028..."
1,IBM,What is cloud computing?,Cloud computing is on-demand access to computi...,"[-0.007971754, -0.047786083, -0.034650102, -0...."
2,IBM,"Join over 100,000 subscribers who read the lat...",Stay up to date on the most important—and intr...,"[0.026251141, 0.067528315, -0.0366205, 0.01086..."
3,IBM,Generic,"You are subscribed. \nIn simpler terms, the ""...","[0.038224988, -0.033074755, -0.0130573325, -0...."
4,IBM,Generic,"The cloud computing model gives you, the custo...","[0.015152888, -0.026298447, -0.055229854, -0.0..."
...,...,...,...,...
1325,Accenture,Generic,"With over 15 years of education excellence, Ta...","[0.012175547, -0.04629302, -0.016613562, -0.02..."
1326,Accenture,Generic,TalentSprint partners with leading enterprises...,"[-0.03034274, -0.05648674, -0.029337484, -0.02..."
1327,Accenture,Generic,Advance your career with the right program \n...,"[0.016358033, 0.038950723, -0.044970207, -0.04..."
1328,Accenture,Generic,A manager relies on it to summarize reports be...,"[0.057241898, 0.026751747, -0.051778775, -0.02..."


In [7]:
# Query a vector
Query = "There are many service providers"
# Query = "Where are the servers located?"
# Query = "What shall be the deciding factor for my embedded system?"

Query_Vector = Embedder_2.encode (Query).tolist ()

**Different Search**  
The search with similarity and top_k always returns a result. This is because criteria is only about top_k  
Whereas setting a distance threshold can identify really meaningful matches.

In [8]:
print ("\nSimilarity : Top k :")
Results = table.search(Query_Vector).distance_type("cosine").limit(5).to_list ()

for Rs in Results :

    print (Rs['_distance'],Rs['source']," ## ",Rs ['text'])

print ("\nSimilarity : Distance threshold then Top k :")
Results = table.search(Query_Vector).distance_type("cosine").distance_range(upper_bound=0.6).limit(5).to_list ()

for Rs in Results :

    print (Rs['_distance'],Rs['source']," ## ",Rs ['text'])


Similarity : Top k :
0.42070508003234863 Oracle  ##  You can choose from several different types of cloud offerings and may, in fact, use many services to meet your needs. These offering might be from the same provider, or you may choose to use multiple cloud providers for different services
0.4290342926979065 Microsoft  ##  These services work with the existing , operating systems, and security frameworks that organizations already use. Typically, organizations combine multiple services to create comprehensive edge solutions
0.43789029121398926 Oracle  ##  Instead of owning their own IT infrastructure or systems, companies access the resources they need from these providers, typically paying only for what they use. Providers may keep costs lower by leveraging economies of scale
0.457297146320343 Fortinet  ##  Further, a telecom can set up a distributed cloud that links a series of on-premises servers designed to support complex edge computing setups
0.4651230573654175 IBM  ##  In the

**Meta Data filtering**  
> The meta data fields present in th vector DB can be used to pre filter the content. Since the meta data provides the broad meaning of the content, it can be a good reference to narrow down  
> One possibility of meta data filtering is to 

In [10]:
# Get all the sources and topics.
DF = table.to_pandas ()
Sources = DF['source'].unique ().tolist ()
Topics = DF['topic'].unique ().tolist ()
Topics_Emb = Embedder_1.encode(Topics, normalize_embeddings=True)

> Formulate the search query that can be used in vector DB filtering

In [11]:
filter = Match_Meta_Data (Query, Topics, Topics_Emb, 3)
filter_text = "(topic IN (" + (",".join(f"'{x}'" for x in filter)) + "))"
filter_text

"(topic IN ('PaaS (Platform-as-a-Service)','IaaS (Infrastructure-as-a-Service)','Telecommunications'))"

**Filter + Search**  
> Filtering criteria applied. Based on the outcome (the chunks that are filtered out), then semantic search is applied  

In [12]:
print ("\n Meta Data Filtered : Top k :")

Results = table.search(Query_Vector).where (filter_text).distance_type("cosine").limit(5).to_list ()

for Rs in Results :

    print (Rs['_distance'],Rs['source']," ## ",Rs ['text'])


 Meta Data Filtered : Top k :
0.457297146320343 Fortinet  ##  Further, a telecom can set up a distributed cloud that links a series of on-premises servers designed to support complex edge computing setups
0.4982035756111145 IBM  ##  Platform as a service (PaaS)  
With PaaS the cloud provider hosts everything at their data center. These include servers, networks, storage, operating system software, and databases
0.5128991007804871 IBM  ##  provides on-demand access to fundamental computing resources—physical and virtual servers, networking and storage—over the internet on a pay-as-you-go basis
0.5258187055587769 Fortinet  ##  Telecoms have been and will likely continue to be one of the most prominent beneficiaries and providers of edge computing
0.5519039034843445 IBM  ##  Infrastructure as a service (IaaS)  
IaaS enables users to scale and shrink resources on an as-needed basis, reducing the need for high up-front capital expenditures or unnecessary on-premises or “owned” infrastructu

In [ ]:
Results

#### Augmentation + Generation

**Meta data Overview**  
Use LLM to create a summary of the content present in the meta data

In [14]:
# Instruction for the LLM
Instruction = """You will be provided a list of topics. Make a summary what is the being discussed, based on the list of topics.
                 Summary in 100 words. Just provide summary text. **No additional Text**
            """
Topic_List  = "Topics : \n"+str(Topics)

   
response = client.models.generate_content(
               #  model="gemini-2.5-flash",
                model="gemini-2.5-flash-lite",
                config =types.GenerateContentConfig(
                            system_instruction=Instruction,
                            # temperature=0.0
                            ),
                contents =Topic_List
)

print(response.text)

This comprehensive overview explores cloud computing, its origins, components, various service models (IaaS, PaaS, SaaS), and deployment types (public, private, hybrid, multicloud). It delves into cloud security, sustainability, and diverse use cases like scaling infrastructure and enabling business continuity. The text also covers the Internet of Things (IoT), explaining its workings, applications in smart devices, homes, and cities, and how it accelerates innovation. Edge computing is examined, highlighting its definition, benefits like reduced latency, and use cases in autonomous vehicles and healthcare. Machine learning, quantum computing, data science, and agentic AI are also discussed, detailing their principles, benefits, challenges, and implementation strategies.


**Prepare Context**  
Find relevant information to the query from Vector DB. Search is done in multiple methods and results consolidated  
Then its made into context information for the LLM to answer

In [15]:
# Query = "What is Cloud Computing?"
Query = "How does IoT get benefitted by Edge Technology?"
# Query = "Is GPU Mandatory for AI?"
# Query = "How can I estimate the Cloud infra cost for my project?"
# Query = "Is Cloud computing cost effective?"

filter = Match_Meta_Data (Query, Topics, Topics_Emb, 3)
filter_text = "(topic IN (" + (",".join(f"'{x}'" for x in filter)) + "))"

Query_Vector = Embedder_2.encode (Query).tolist ()

# Typical Similarity Seach with threshold
Results_1 = table.search(Query_Vector).distance_type("cosine").distance_range(upper_bound=0.6).limit(5).to_list ()
print (len(Results_1))

# Search with meta data filtering
Results_2 = table.search(Query_Vector).where (filter_text).distance_type("cosine").limit(5).to_list ()
print (len(Results_2))

# Remove Duplicates
Context = [d['text'] for d in Results_1]
Context = Context + [d['text'] for d in Results_2]

Context = list(set(Context))
print (len(Context))

5
5
10


In [16]:
# Instruction for the LLM
Instruction = """You will be given context information and a user query. You have to provide an answer to user query based on information provided in context.
                Answer **ONLY** based on context. If sufficient details are not in context, respond as "No Sufficient Details"
            """
   
response = client.models.generate_content(
                # model="gemini-2.5-flash",
                model='gemini-2.5-flash-lite',
                config =types.GenerateContentConfig(
                            system_instruction=Instruction,
                            # temperature=0.0
                            ),
                contents = ["Context : \n"+str(Context), "User Query : \n"+Query]
)

print(response.text)

Edge computing benefits IoT by increasing the computing power at the edges of an IoT network, which reduces communication latency and improves response times. This allows IoT devices to process data locally, enabling faster access to insights, immediate actions, and reduced transmission costs by filtering unnecessary data. Examples include industrial applications like smart cities and robots, as well as consumer devices like smartphones and home security controls.


In [17]:
  
response = client.models.generate_content(
                # model="gemini-2.5-flash",
                model='gemini-2.5-flash-lite',
                config =types.GenerateContentConfig(
                            system_instruction="Respond to User query",
                            ),
                contents = "User Query : \n"+Query
)

print(response.text)

User Query : How does IoT get benefitted by Edge Technology?

Edge technology significantly enhances the capabilities and efficiency of the Internet of Things (IoT) by bringing processing and data analysis closer to the data source. Here's a breakdown of the key benefits:

**1. Reduced Latency and Real-time Processing:**

*   **How it works:** Instead of sending all data to a distant cloud server for processing, edge devices (like gateways, routers, or even powerful sensors) perform initial analysis and decision-making locally.
*   **Benefit for IoT:** This is crucial for time-sensitive IoT applications such as:
    *   **Autonomous vehicles:** Millisecond-level decisions are needed for safety.
    *   **Industrial automation:** Immediate response to machine failures or anomalies.
    *   **Healthcare:** Real-time monitoring of patients and critical alerts.
    *   **Smart grids:** Rapid response to power fluctuations.

**2. Bandwidth Optimization and Cost Savings:**

*   **How it work